# 02 Train, evaluate and ablate (Stage B)

Built from repo commit `20b7eac`. Trains every experiment in `configs/experiments.yaml` on the cached Stage A features (notebook 01), on two GPUs, resumable. Then builds all result tables and figures with `src/analysis/report.py`. Every number shown is produced by this run.

In [ ]:
import base64, io, zipfile, os, sys, glob, json, subprocess
BUNDLE = '...'
CODE = '/kaggle/tmp/code'; os.makedirs(CODE, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall(CODE)
sys.path.insert(0, CODE)
def find(p):
    h = glob.glob(p, recursive=True)
    return h[0] if h else None
FEAT = os.path.dirname(find('/kaggle/input/**/features/ccd_pos_000001.npz'))
CCD_ROOT = os.path.dirname(find('/kaggle/input/**/CCD/Crash-1500.txt'))
DOTA_META = os.path.dirname(find('/kaggle/input/**/dota_official/metadata_train.json'))
print(FEAT, CCD_ROOT, DOTA_META, len(glob.glob(FEAT + '/*.npz')), 'feature files')

In [ ]:
OUT = '/kaggle/working'
os.makedirs(OUT + '/runs', exist_ok=True)
# resume: copy finished runs from an attached earlier run of this notebook
for f in glob.glob('/kaggle/input/**/runs/*/*/*/metrics.json', recursive=True):
    src = os.path.dirname(f); dst = os.path.join(OUT, 'runs', *src.split('/runs/')[1].split('/'))
    if not os.path.exists(dst): subprocess.run(['cp', '-r', src, dst])
import torch
n = max(1, torch.cuda.device_count())
env = dict(os.environ, PYTHONPATH=CODE)
procs = [subprocess.Popen([sys.executable, '-u', '-m', 'src.train.run', '--feat_dir', FEAT, '--ccd_root', CCD_ROOT,
                           '--dota_meta', DOTA_META, '--out', OUT, '--worker', str(i), '--nworkers', str(n), '--budget_h', '7.6'],
                          cwd=CODE, env=dict(env, CUDA_VISIBLE_DEVICES=str(i)), stdout=open(f'{OUT}/train_gpu{i}.log', 'w'),
                          stderr=subprocess.STDOUT) for i in range(n)]
for p in procs: p.wait()
for i in range(n): print(''.join(open(f'{OUT}/train_gpu{i}.log').readlines()[-15:]))

In [ ]:
r = subprocess.run([sys.executable, '-m', 'src.analysis.shortcut', '--feat_dir', FEAT, '--ccd_root', CCD_ROOT,
                    '--dota_meta', DOTA_META, '--out', OUT + '/shortcut_probe.csv'],
                   cwd=CODE, env=dict(os.environ, PYTHONPATH=CODE), capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-2000:])

In [ ]:
r = subprocess.run([sys.executable, '-m', 'src.analysis.report', '--runs', OUT + '/runs', '--out', OUT + '/results'],
                   cwd=CODE, env=dict(os.environ, PYTHONPATH=CODE), capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-3000:])
import shutil
if os.path.exists(OUT + '/shortcut_probe.csv'): shutil.copy(OUT + '/shortcut_probe.csv', OUT + '/results/tables/')
for f in glob.glob(OUT + '/leakage_*.json'): print(os.path.basename(f), open(f).read())

## Results tables

In [ ]:
from IPython.display import Markdown, Image, display
for n in ('main_comparable', 'main_and_ablations_clean', 'cross_dataset'):
    f = f'{OUT}/results/tables/{n}.md'
    if os.path.exists(f): display(Markdown(f'### {n}\n' + open(f).read()))
import pandas as pd
for n in ('early_late_summary.csv', 'errors_ccd_source.csv', 'errors_dota_source.csv'):
    f = f'{OUT}/results/tables/{n}'
    if os.path.exists(f): display(Markdown(f'### {n}')); display(pd.read_csv(f))
for f in sorted(glob.glob(OUT + '/results/figures/*.png')): display(Image(f, width=1000))